# Notebook note

The saved ablation JSON is enough to remake the paper figure. Rerun this notebook when changing where dropout is applied inside the transformer block.


# ViT component ablations

This ablation separates dropout in attention, in the MLP sub-block, and in both places. The point is to check whether the scheduling effect is tied to one part of the residual block or remains visible across the full block.


In [ ]:
from pathlib import Path
import sys

for _root in [Path.cwd(), *Path.cwd().parents]:
    if (_root / "src" / "dropout_mft").exists():
        sys.path.insert(0, str(_root / "src"))
        break

import torch
import torch.nn as nn
import torch.optim as optim
from torch.optim.lr_scheduler import CosineAnnealingLR
from torch.cuda.amp import autocast, GradScaler
from torchvision import datasets, transforms
import numpy as np
import matplotlib.pyplot as plt
import matplotlib as mpl
from tqdm.auto import tqdm
import json
import warnings
warnings.filterwarnings('ignore')

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Device: {device}")

if device == 'cuda':
    torch.backends.cudnn.benchmark = True
    torch.backends.cuda.matmul.allow_tf32 = True
    print(f"GPU: {torch.cuda.get_device_name(0)}")

USE_AMP = device == 'cuda'
AMP_DTYPE = torch.bfloat16 if (device == 'cuda' and torch.cuda.is_bf16_supported()) else torch.float16

In [ ]:
# Settings

N_SIMULATIONS = 5

# Dropout
H_BAR = 0.1
H_MAX = 0.2
SCHEDULES = ['none', 'constant', 'reverse_step']

# Where dropout is applied
ABLATION_MODES = ['both', 'attn_only', 'mlp_only']

# ViT Architecture
EMBED_DIM = 128
DEPTH = 12
NUM_HEADS = 16
MLP_RATIO = 4.0
PATCH_SIZE = 4

# Training
EPOCHS = 75
BATCH_SIZE = 1000
LEARNING_RATE = 5e-4
WEIGHT_DECAY = 0.00
LR_MIN = 5e-6

# Evaluate every N epochs
EVAL_EVERY = 1

# Dataset
USE_FULL_DATASET = True
TRAIN_SIZE = None if USE_FULL_DATASET else 5000
TEST_SIZE = None if USE_FULL_DATASET else 5000

In [ ]:
plt.style.use('seaborn-v0_8-whitegrid')

from dropout_mft.schedules import DISPLAY_NAMES as SCHED_NAMES
from dropout_mft.style import (
    ABLATION_MODE_COLORS,
    ABLATION_MODE_NAMES as MODE_NAMES,
    ABLATION_MODE_STYLES as MODE_STYLES,
    SCHEDULE_COLORS as SCHED_COLORS,
    schedule_color,
)

def get_sched_color(s):
    return schedule_color(s)

def get_mode_style(m):
    return MODE_STYLES.get(m, '-')

def get_mode_name(m):
    return MODE_NAMES.get(m, m)

def get_sched_name(s):
    return SCHED_NAMES.get(s, s)

In [ ]:
# Dropout schedules

def get_dropout_schedule(sched, depth, h_bar, h_max=None):
    if sched == 'none': return [0.0] * depth
    if sched == 'constant': return [h_bar] * depth
    if sched == 'linear':
        return [2*h_bar * i / (depth - 1) for i in range(depth)] if depth > 1 else [h_bar]
    if sched == 'reverse_linear':
        return [2*h_bar * (depth-1-i) / (depth-1) for i in range(depth)] if depth > 1 else [h_bar]
    if sched == 'step':
        h_max = h_max or 2*h_bar
        n = max(1, int(np.ceil(h_bar/h_max * depth)))
        return [0.0]*(depth-n) + [h_bar*depth/n]*n
    if sched == 'reverse_step':
        h_max = h_max or 2*h_bar
        n = max(1, int(np.ceil(h_bar/h_max * depth)))
        return [h_bar*depth/n]*n + [0.0]*(depth-n)
    raise ValueError(sched)

# Precompute schedules
theory = {s: {'h_layers': get_dropout_schedule(s, DEPTH, H_BAR, H_MAX)} for s in SCHEDULES}
for s in SCHEDULES:
    print(f"{get_sched_name(s):22s}: mean={np.mean(theory[s]['h_layers']):.4f}, schedule={theory[s]['h_layers'][:4]}...")

In [ ]:
# Vision Transformer with ablations

class PatchEmbed(nn.Module):
    def __init__(self, img_size=32, patch_size=PATCH_SIZE, embed_dim=384):
        super().__init__()
        self.proj = nn.Conv2d(3, embed_dim, patch_size, patch_size)
        self.num_patches = (img_size // patch_size) ** 2
    def forward(self, x):
        return self.proj(x).flatten(2).transpose(1, 2)

class Attention(nn.Module):
    def __init__(self, dim, heads=8):
        super().__init__()
        self.heads, self.scale = heads, (dim // heads) ** -0.5
        self.qkv = nn.Linear(dim, dim * 3, bias=False)
        self.proj = nn.Linear(dim, dim)
    def forward(self, x):
        B, N, C = x.shape
        qkv = self.qkv(x).reshape(B, N, 3, self.heads, C // self.heads).permute(2,0,3,1,4)
        q, k, v = qkv.unbind(0)
        x = torch.nn.functional.scaled_dot_product_attention(q, k, v)
        return self.proj(x.transpose(1,2).reshape(B, N, C))

class MLP(nn.Module):
    def __init__(self, dim, ratio=4.):
        super().__init__()
        h = int(dim * ratio)
        self.fc1, self.fc2 = nn.Linear(dim, h), nn.Linear(h, dim)
        self.act = nn.ReLU()
    def forward(self, x):
        return self.fc2(self.act(self.fc1(x)))

class Block(nn.Module):
    """Transformer block with separate dropout for attention and MLP.

    Args:
        dim: embedding dimension
        heads: number of attention heads
        ratio: MLP expansion ratio
        drop_attn: dropout rate for attention sublayer (post-attention residual)
        drop_mlp: dropout rate for MLP sublayer (post-MLP residual)
    """
    def __init__(self, dim, heads, ratio=4., drop_attn=0., drop_mlp=0.):
        super().__init__()
        self.norm1, self.norm2 = nn.LayerNorm(dim), nn.LayerNorm(dim)
        self.attn, self.mlp = Attention(dim, heads), MLP(dim, ratio)
        self.drop_attn = nn.Dropout(drop_attn)
        self.drop_mlp = nn.Dropout(drop_mlp)

    def forward(self, x):
        x = x + self.drop_attn(self.attn(self.norm1(x)))
        x = x + self.drop_mlp(self.mlp(self.norm2(x)))
        return x

class ViT(nn.Module):
    """Vision Transformer with ablation support for dropout location.

    Args:
        depth: number of transformer blocks
        dim: embedding dimension
        heads: number of attention heads
        ratio: MLP expansion ratio
        h_layers: list of dropout rates per layer
        ablation_mode: 'both', 'attn_only', or 'mlp_only'
    """
    def __init__(self, depth=12, dim=384, heads=6, ratio=4., h_layers=None, ablation_mode='both'):
        super().__init__()
        self.patch = PatchEmbed(32, PATCH_SIZE, dim)
        n = self.patch.num_patches
        self.cls = nn.Parameter(torch.zeros(1, 1, dim))
        self.pos = nn.Parameter(torch.zeros(1, n + 1, dim))
        h_layers = h_layers or [0.]*depth

        # Blocks with ablation-specific dropout
        blocks = []
        for i in range(depth):
            h = h_layers[i]
            if ablation_mode == 'both':
                drop_attn, drop_mlp = h, h
            elif ablation_mode == 'attn_only':
                drop_attn, drop_mlp = h, 0.0
            elif ablation_mode == 'mlp_only':
                drop_attn, drop_mlp = 0.0, h
            else:
                raise ValueError(f"Unknown ablation_mode: {ablation_mode}")
            blocks.append(Block(dim, heads, ratio, drop_attn, drop_mlp))
        self.blocks = nn.ModuleList(blocks)

        self.norm = nn.LayerNorm(dim)
        self.head = nn.Linear(dim, 10)
        nn.init.trunc_normal_(self.pos, std=.02)
        nn.init.trunc_normal_(self.cls, std=.02)
        self.apply(self._init)

    def _init(self, m):
        if isinstance(m, nn.Linear):
            nn.init.trunc_normal_(m.weight, std=.02)
            if m.bias is not None: nn.init.zeros_(m.bias)
        elif isinstance(m, nn.LayerNorm):
            nn.init.ones_(m.weight); nn.init.zeros_(m.bias)

    def forward(self, x):
        x = self.patch(x)
        x = torch.cat([self.cls.expand(x.size(0),-1,-1), x], 1) + self.pos
        for blk in self.blocks: x = blk(x)
        return self.head(self.norm(x)[:,0])

# Quick parameter-count check
print(f"Params: {sum(p.numel() for p in ViT(DEPTH, EMBED_DIM, NUM_HEADS, MLP_RATIO).parameters()):,}")

In [ ]:
# Data kept on the GPU

def iterate_batches(x, y, bs, shuffle=True):
    n = x.shape[0]
    idx = torch.randperm(n, device=x.device) if shuffle else torch.arange(n, device=x.device)
    for i in range(0, n, bs):
        yield x[idx[i:i+bs]], y[idx[i:i+bs]]

def get_cifar10(train_size=None, test_size=None):
    mean = torch.tensor([0.4914, 0.4822, 0.4465]).view(1,3,1,1)
    std = torch.tensor([0.2470, 0.2435, 0.2616]).view(1,3,1,1)

    train = datasets.CIFAR10('./data', train=True, download=True)
    test = datasets.CIFAR10('./data', train=False, download=True)

    rng = np.random.RandomState(0)
    tr_idx = rng.choice(len(train.data), train_size, replace=False) if train_size else np.arange(len(train.data))
    te_idx = rng.choice(len(test.data), test_size, replace=False) if test_size else np.arange(len(test.data))

    x_tr = torch.from_numpy(train.data[tr_idx]).permute(0,3,1,2).float().div_(255)
    y_tr = torch.tensor(np.array(train.targets)[tr_idx])
    x_te = torch.from_numpy(test.data[te_idx]).permute(0,3,1,2).float().div_(255)
    y_te = torch.tensor(np.array(test.targets)[te_idx])

    if device == 'cuda':
        x_tr, y_tr, x_te, y_te = x_tr.cuda(), y_tr.cuda(), x_te.cuda(), y_te.cuda()
        mean, std = mean.cuda(), std.cuda()

    x_tr, x_te = (x_tr - mean) / std, (x_te - mean) / std
    print(f"Train: {x_tr.shape[0]}, Test: {x_te.shape[0]}")
    return (x_tr, y_tr), (x_te, y_te)

train_data, test_data = get_cifar10(TRAIN_SIZE, TEST_SIZE)

In [ ]:
# Training code

def train_epoch(model, data, opt, crit, bs):
    model.train()
    x, y = data
    total, correct, loss_sum = 0, 0, 0.
    for xb, yb in iterate_batches(x, y, bs):
        opt.zero_grad(set_to_none=True)
        with autocast(dtype=AMP_DTYPE, enabled=USE_AMP):
            out = model(xb)
            loss = crit(out, yb)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        opt.step()
        total += xb.size(0)
        correct += (out.argmax(1) == yb).sum().item()
        loss_sum += loss.item() * xb.size(0)
    return loss_sum/total, 100*correct/total

@torch.no_grad()
def evaluate(model, data, crit, bs):
    model.eval()
    x, y = data
    total, correct, loss_sum = 0, 0, 0.
    for xb, yb in iterate_batches(x, y, bs, shuffle=False):
        with autocast(dtype=AMP_DTYPE, enabled=USE_AMP):
            out = model(xb)
            loss = crit(out, yb)
        total += xb.size(0)
        correct += (out.argmax(1) == yb).sum().item()
        loss_sum += loss.item() * xb.size(0)
    return loss_sum/total, 100*correct/total

In [ ]:
import wandb
wandb.login()

In [ ]:
def run_ablation_experiments(schedules, ablation_modes, n_sims, epochs):
    """Run experiments for all schedule × ablation_mode combinations with W&B tracking."""
    results = {}

    for mode in ablation_modes:
        for sched in schedules:
            # No-dropout is the same across ablation modes.
            if sched == 'none' and mode != 'both':
                continue

            key = f"{mode}_{sched}"
            print(f"\n{'='*60}")
            print(f"Ablation: {get_mode_name(mode)} | Schedule: {get_sched_name(sched)}")
            print(f"{'='*60}")

            h_layers = theory[sched]['h_layers']
            histories = []

            for sim in range(n_sims):
                # --- W&B INIT ---
                # One W&B run per seed
                run = wandb.init(
                    project="vit-dropout-ablation",
                    group=key,  # Groups all seeds for this config together
                    name=f"{key}_sim{sim}",
                    reinit=True,
                    config={
                        "ablation_mode": mode,
                        "schedule_type": sched,
                        "h_bar": H_BAR,
                        "h_max": H_MAX,
                        "epochs": epochs,
                        "batch_size": BATCH_SIZE,
                        "lr": LEARNING_RATE,
                        "depth": DEPTH,
                        "embed_dim": EMBED_DIM,
                        "h_layers": h_layers, # Logs the exact schedule list
                        "seed": 42 + sim
                    }
                )

                # Seed this run
                seed = 42 + sim
                torch.manual_seed(seed)
                np.random.seed(seed)

                model = ViT(DEPTH, EMBED_DIM, NUM_HEADS, MLP_RATIO,
                           h_layers, ablation_mode=mode).to(device)

                # Uncomment if you want W&B to watch the model
                # wandb.watch(model, log="all", log_freq=10)

                opt = optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
                scheduler = CosineAnnealingLR(opt, T_max=epochs, eta_min=LR_MIN)
                crit = nn.CrossEntropyLoss()

                hist = {'train_acc': [], 'test_acc': [], 'train_loss': [], 'test_loss': []}
                best_test_acc = 0.

                pbar = tqdm(range(epochs), desc=f'{mode}/{sched} sim {sim+1}/{n_sims}', leave=True)

                for ep in pbar:
                    tr_loss, tr_acc = train_epoch(model, train_data, opt, crit, BATCH_SIZE)
                    scheduler.step()

                    if ep % EVAL_EVERY == 0 or ep == epochs - 1:
                        te_loss, te_acc = evaluate(model, test_data, crit, BATCH_SIZE)
                    else:
                        te_loss = hist['test_loss'][-1] if hist['test_loss'] else 0
                        te_acc = hist['test_acc'][-1] if hist['test_acc'] else 0

                    if te_acc > best_test_acc: best_test_acc = te_acc

                    # --- W&B LOGGING ---
                    wandb.log({
                        "epoch": ep,
                        "train/loss": tr_loss,
                        "train/acc": tr_acc,
                        "test/loss": te_loss,
                        "test/acc": te_acc,
                        "test/best_acc": best_test_acc,
                        "lr": opt.param_groups[0]['lr'],
                        # Generalization gap
                        "gen_gap": tr_acc - te_acc
                    })

                    hist['train_acc'].append(tr_acc)
                    hist['test_acc'].append(te_acc)
                    hist['train_loss'].append(tr_loss)
                    hist['test_loss'].append(te_loss)

                    pbar.set_postfix({'tr_L': f'{tr_loss:.3f}', 'tr_A': f'{tr_acc:.1f}',
                                     'te_A': f'{te_acc:.1f}', 'best': f'{best_test_acc:.1f}'})

                histories.append(hist)

                # --- W&B FINISH ---
                # Close this W&B run before starting the next one
                wandb.finish()

                del model
                if device == 'cuda': torch.cuda.empty_cache()

            # Aggregate curves across seeds
            results[key] = {
                'mode': mode,
                'schedule': sched,
                'h_layers': h_layers,
                'train_acc': np.array([h['train_acc'] for h in histories]),
                'test_acc': np.array([h['test_acc'] for h in histories]),
                'train_loss': np.array([h['train_loss'] for h in histories]),
                'test_loss': np.array([h['test_loss'] for h in histories]),
            }

            final_test = results[key]['test_acc'][:, -1]
            print(f"\n→ Final test acc: {final_test.mean():.2f}% ± {final_test.std():.2f}%")

    return results

# Run all experiments
results = run_ablation_experiments(SCHEDULES, ABLATION_MODES, N_SIMULATIONS, EPOCHS)

In [ ]:
# Save

def save_results(results, filename='ablation_results.json'):
    save_data = {}
    for key, r in results.items():
        save_data[key] = {
            'mode': r['mode'],
            'schedule': r['schedule'],
            'h_layers': r['h_layers'],
            'train_acc': r['train_acc'].tolist(),
            'test_acc': r['test_acc'].tolist(),
            'train_loss': r['train_loss'].tolist(),
            'test_loss': r['test_loss'].tolist(),
        }
    with open(filename, 'w') as f:
        json.dump(save_data, f, indent=2)
    print(f"Results saved to {filename}")

save_results(results)

In [ ]:
import json
import numpy as np

with open('ablation_results.json', 'r') as f:
    data = json.load(f)

results = {key: {k: np.array(v) if isinstance(v, list) else v for k, v in r.items()} for key, r in data.items()}

In [ ]:
# Summary table

print("\n" + "="*80)
print("ABLATION STUDY SUMMARY")
print("="*80)
print(f"{'Ablation Mode':<20} {'Schedule':<15} {'Final Test Acc':>20} {'Best Test Acc':>18}")
print("-"*80)

for key, r in sorted(results.items()):
    final = r['test_acc'][:, -1]
    best = r['test_acc'].max(axis=1)
    n = len(final)
    print(f"{get_mode_name(r['mode']):<20} {get_sched_name(r['schedule']):<15} "
          f"{final.mean():>8.2f}% ± {final.std()/np.sqrt(n):>4.2f}%   {best.mean():>8.2f}% ± {best.std()/np.sqrt(n):>4.2f}%")

print("="*80)
print("Note: Errors are SEM (N=5)")

In [ ]:
# Test accuracy by ablation mode

fig, axes = plt.subplots(1, 3, figsize=(15, 4.5), sharey=True)

for ax, mode in zip(axes, ABLATION_MODES):
    ax.set_title(get_mode_name(mode), fontsize=12, fontweight='bold')

    for sched in SCHEDULES:
        key = f"{mode}_{sched}"
        if key not in results:
            # For 'none' schedule in attn_only/mlp_only, use the both_none result
            if sched == 'none':
                key = 'both_none'
            else:
                continue

        r = results[key]
        test_acc = r['test_acc']
        mean = test_acc.mean(axis=0)
        std = test_acc.std(axis=0)
        epochs = np.arange(1, len(mean) + 1)

        color = get_sched_color(sched)
        label = get_sched_name(sched)
        if sched == 'none' and mode != 'both':
            label += ' (baseline)'

        ax.plot(epochs, mean, color=color, linewidth=2, label=label)
        ax.fill_between(epochs, mean - std, mean + std, color=color, alpha=0.15)

    ax.set_xlabel('Epoch')
    ax.legend(loc='lower right', fontsize=9)
    ax.grid(True, alpha=0.3)

axes[0].set_ylabel('Test Accuracy (%)')
fig.suptitle('Dropout Scheduling Ablation: Where Does Dropout Help?', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('ablation_by_mode.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Scheduling effect by location

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))

# Plot 2a: Compare constant dropout across ablation modes
ax = axes[0]
ax.set_title('Constant Dropout: Effect of Location', fontsize=11, fontweight='bold')

for mode in ABLATION_MODES:
    key = f"{mode}_constant"
    if key not in results:
        continue
    r = results[key]
    test_acc = r['test_acc']
    mean = test_acc.mean(axis=0)
    std = test_acc.std(axis=0)
    epochs = np.arange(1, len(mean) + 1)

    style = get_mode_style(mode)
    ax.plot(epochs, mean, linestyle=style, color='#EE6677', linewidth=2, label=get_mode_name(mode))
    ax.fill_between(epochs, mean - std, mean + std, color='#EE6677', alpha=0.1)

# No-dropout baseline
r = results['both_none']
mean = r['test_acc'].mean(axis=0)
ax.plot(np.arange(1, len(mean)+1), mean, '--', color='#4477AA', linewidth=2, label='No dropout', alpha=0.7)

ax.set_xlabel('Epoch')
ax.set_ylabel('Test Accuracy (%)')
ax.legend(loc='lower right', fontsize=9)
ax.grid(True, alpha=0.3)

# Plot 2b: Compare scheduled dropout across ablation modes
ax = axes[1]
ax.set_title('Scheduled Dropout (Early): Effect of Location', fontsize=11, fontweight='bold')

for mode in ABLATION_MODES:
    key = f"{mode}_reverse_step"
    if key not in results:
        continue
    r = results[key]
    test_acc = r['test_acc']
    mean = test_acc.mean(axis=0)
    std = test_acc.std(axis=0)
    epochs = np.arange(1, len(mean) + 1)

    style = get_mode_style(mode)
    ax.plot(epochs, mean, linestyle=style, color='#AA3377', linewidth=2, label=get_mode_name(mode))
    ax.fill_between(epochs, mean - std, mean + std, color='#AA3377', alpha=0.1)

# No-dropout baseline
r = results['both_none']
mean = r['test_acc'].mean(axis=0)
ax.plot(np.arange(1, len(mean)+1), mean, '--', color='#4477AA', linewidth=2, label='No dropout', alpha=0.7)

ax.set_xlabel('Epoch')
ax.legend(loc='lower right', fontsize=9)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('ablation_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Bar summary

fig, ax = plt.subplots(figsize=(10, 5))

# Organize data for bar chart
schedules_to_plot = ['constant', 'reverse_step']
x = np.arange(len(schedules_to_plot))
width = 0.25

# Get baseline (no dropout)
baseline = results['both_none']['test_acc'][:, -1].mean()

for i, mode in enumerate(ABLATION_MODES):
    means = []
    stds = []
    for sched in schedules_to_plot:
        key = f"{mode}_{sched}"
        if key in results:
            final = results[key]['test_acc'][:, -1]
            means.append(final.mean())
            stds.append(final.std())
        else:
            means.append(0)
            stds.append(0)

    offset = (i - 1) * width
    bars = ax.bar(x + offset, means, width, yerr=stds,
                  label=get_mode_name(mode), capsize=3, alpha=0.85)

# Baseline line
ax.axhline(y=baseline, color='#4477AA', linestyle='--', linewidth=2,
           label=f'No dropout ({baseline:.1f}%)', alpha=0.7)

ax.set_xlabel('Dropout Schedule', fontsize=11)
ax.set_ylabel('Final Test Accuracy (%)', fontsize=11)
ax.set_title('Ablation Study: Where Should Dropout Be Applied?', fontsize=13, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels([get_sched_name(s) for s in schedules_to_plot])
ax.legend(loc='lower right')
ax.grid(True, alpha=0.3, axis='y')

# Set y-axis to focus on differences
all_finals = [results[k]['test_acc'][:, -1].mean() for k in results]
ax.set_ylim(min(all_finals) - 2, max(all_finals) + 2)

plt.tight_layout()
plt.savefig('ablation_bars.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Generalization gap

fig, ax = plt.subplots(figsize=(10, 5))

for mode in ABLATION_MODES:
    for sched in ['constant', 'reverse_step']:
        key = f"{mode}_{sched}"
        if key not in results:
            continue
        r = results[key]

        # Generalization gap = train_acc - test_acc
        gap = r['train_acc'] - r['test_acc']
        mean_gap = gap.mean(axis=0)
        std_gap = gap.std(axis=0)
        epochs = np.arange(1, len(mean_gap) + 1)

        style = get_mode_style(mode)
        color = get_sched_color(sched)
        label = f"{get_mode_name(mode)} - {get_sched_name(sched)}"

        ax.plot(epochs, mean_gap, linestyle=style, color=color, linewidth=2, label=label)

# No-dropout baseline
r = results['both_none']
gap = r['train_acc'] - r['test_acc']
mean_gap = gap.mean(axis=0)
ax.plot(np.arange(1, len(mean_gap)+1), mean_gap, '--', color='#4477AA',
        linewidth=2, label='No dropout', alpha=0.7)

ax.set_xlabel('Epoch', fontsize=11)
ax.set_ylabel('Generalization Gap (Train - Test Acc)', fontsize=11)
ax.set_title('Generalization Gap by Dropout Location', fontsize=13, fontweight='bold')
ax.legend(loc='upper left', fontsize=8, ncol=2)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('ablation_gen_gap.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Significance checks

from scipy import stats

print("\n" + "="*80)
print("STATISTICAL SIGNIFICANCE TESTS (Welch's t-test, final epoch)")
print("="*80)

# Compare scheduling vs constant for each ablation mode
print("\n1. Scheduling (reverse_step) vs Constant - within each location:")
print("-"*60)
for mode in ABLATION_MODES:
    key_const = f"{mode}_constant"
    key_sched = f"{mode}_reverse_step"
    if key_const not in results or key_sched not in results:
        continue

    const = results[key_const]['test_acc'][:, -1]
    sched = results[key_sched]['test_acc'][:, -1]
    t_stat, p_val = stats.ttest_ind(sched, const, equal_var=False)

    diff = sched.mean() - const.mean()
    sig = "*" if p_val < 0.05 else ""
    sig += "*" if p_val < 0.01 else ""
    sig += "*" if p_val < 0.001 else ""

    print(f"{get_mode_name(mode):<20}: Δ = {diff:+.2f}%, p = {p_val:.4f} {sig}")

# Compare ablation modes for scheduled dropout
print("\n2. Comparing dropout locations (with scheduling):")
print("-"*60)
comparisons = [('both', 'attn_only'), ('both', 'mlp_only'), ('attn_only', 'mlp_only')]
for m1, m2 in comparisons:
    key1 = f"{m1}_reverse_step"
    key2 = f"{m2}_reverse_step"
    if key1 not in results or key2 not in results:
        continue

    a1 = results[key1]['test_acc'][:, -1]
    a2 = results[key2]['test_acc'][:, -1]
    t_stat, p_val = stats.ttest_ind(a1, a2, equal_var=False)

    diff = a1.mean() - a2.mean()
    sig = "*" if p_val < 0.05 else ""
    sig += "*" if p_val < 0.01 else ""
    sig += "*" if p_val < 0.001 else ""

    print(f"{get_mode_name(m1)} vs {get_mode_name(m2)}: Δ = {diff:+.2f}%, p = {p_val:.4f} {sig}")

print("\n* p<0.05, ** p<0.01, *** p<0.001")

In [ ]:
# Final summary figure

fig = plt.figure(figsize=(14, 8))

# Layout
gs = fig.add_gridspec(2, 3, hspace=0.3, wspace=0.25)

# Top row: test accuracy by mode
for idx, mode in enumerate(ABLATION_MODES):
    ax = fig.add_subplot(gs[0, idx])
    ax.set_title(get_mode_name(mode), fontsize=11, fontweight='bold')

    for sched in SCHEDULES:
        key = f"{mode}_{sched}"
        if key not in results:
            if sched == 'none':
                key = 'both_none'
            else:
                continue

        r = results[key]
        test_acc = r['test_acc']
        mean = test_acc.mean(axis=0)
        std = test_acc.std(axis=0)
        epochs = np.arange(1, len(mean) + 1)

        color = get_sched_color(sched)
        ax.plot(epochs, mean, color=color, linewidth=1.8, label=get_sched_name(sched))
        ax.fill_between(epochs, mean - std, mean + std, color=color, alpha=0.12)

    ax.set_xlabel('Epoch', fontsize=9)
    if idx == 0:
        ax.set_ylabel('Test Accuracy (%)', fontsize=10)
    ax.legend(loc='lower right', fontsize=8)
    ax.grid(True, alpha=0.3)

# Bottom left: bar comparison
ax = fig.add_subplot(gs[1, 0:2])
schedules_to_plot = ['constant', 'reverse_step']
x = np.arange(len(schedules_to_plot))
width = 0.25
baseline = results['both_none']['test_acc'][:, -1].mean()

colors_mode = [ABLATION_MODE_COLORS[m] for m in ABLATION_MODES]
for i, mode in enumerate(ABLATION_MODES):
    means = []
    stds = []
    for sched in schedules_to_plot:
        key = f"{mode}_{sched}"
        if key in results:
            final = results[key]['test_acc'][:, -1]
            means.append(final.mean())
            stds.append(final.std())
        else:
            means.append(0)
            stds.append(0)

    offset = (i - 1) * width
    ax.bar(x + offset, means, width, yerr=stds, label=get_mode_name(mode),
           capsize=3, alpha=0.85, color=colors_mode[i])

ax.axhline(y=baseline, color='red', linestyle='--', linewidth=1.5,
           label=f'No dropout ({baseline:.1f}%)', alpha=0.8)
ax.set_xlabel('Dropout Schedule', fontsize=10)
ax.set_ylabel('Final Test Accuracy (%)', fontsize=10)
ax.set_title('Final Accuracy Comparison', fontsize=11, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels([get_sched_name(s) for s in schedules_to_plot])
ax.legend(loc='lower right', fontsize=8)
ax.grid(True, alpha=0.3, axis='y')
all_finals = [results[k]['test_acc'][:, -1].mean() for k in results]
ax.set_ylim(min(all_finals) - 1.5, max(all_finals) + 1.5)

# Bottom right: short text summary
ax = fig.add_subplot(gs[1, 2])
ax.axis('off')

# Compute key metrics
both_sched = results.get('both_reverse_step', {}).get('test_acc', np.array([[0]]))[:, -1].mean()
both_const = results.get('both_constant', {}).get('test_acc', np.array([[0]]))[:, -1].mean()
attn_sched = results.get('attn_only_reverse_step', {}).get('test_acc', np.array([[0]]))[:, -1].mean()
mlp_sched = results.get('mlp_only_reverse_step', {}).get('test_acc', np.array([[0]]))[:, -1].mean()

findings = f"""Key Findings:

1. Scheduling benefit (Both):
   {both_sched:.1f}% vs {both_const:.1f}% (Δ={both_sched-both_const:+.1f}%)

2. Location comparison (Scheduled):
   • Both: {both_sched:.1f}%
   • Attn only: {attn_sched:.1f}%
   • MLP only: {mlp_sched:.1f}%

3. Baseline (No dropout):
   {baseline:.1f}%

Config: ViT-{DEPTH}L, {EMBED_DIM}d, {EPOCHS}ep
h̄={H_BAR}, {N_SIMULATIONS} seeds"""

ax.text(0.05, 0.95, findings, transform=ax.transAxes, fontsize=9.5,
        verticalalignment='top', fontfamily='monospace',
        bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))

fig.suptitle('ViT Dropout Scheduling: Ablation Study', fontsize=14, fontweight='bold', y=0.98)
plt.savefig('ablation_summary.png', dpi=150, bbox_inches='tight')
plt.show()